# Options Pricing Engine — Deep Dive

A guided tour of the `options-pricing-engine` library: pricing across four models, the full Greek suite, volatility smile construction, validation, and scenario analysis.

> Run from the repository root so the package imports resolve.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from pricing import BlackScholesMerton, Black76, CRRBinomial, BoyleTrinomial, OptionContract
from greeks import GreekEngine, BSMGreeks, NumericalGreeks
from volatility import VolatilitySmile, SABRModel, SABRParams
from validation import ModelComparison, PutCallParityCheck, ConvergenceAnalyzer
from scenarios import StressTester, GreekPnLAttribution

## 1. Pricing across all four models

For a European contract every model should agree to numerical tolerance.

In [ ]:
contract = OptionContract(S=100, K=100, T=1.0, r=0.05, q=0.0, sigma=0.2)
ModelComparison(n_steps=500).comparison_table(contract).round(5)

## 2. The Greek engine

Closed-form BSM Greeks versus finite differences on the binomial tree.

In [ ]:
analytical = BSMGreeks().compute(100, 100, 1.0, 0.05, 0.2, 'call')
numerical = NumericalGreeks(CRRBinomial(400)).compute(100, 100, 1.0, 0.05, 0.2, 'call')
pd.DataFrame({
    'analytical_BSM': analytical.as_dict(),
    'numerical_tree': numerical.as_dict(),
}).round(5)

## 3. Tree convergence

Both lattices converge to the analytical benchmark as the step count grows.

In [ ]:
analysis = ConvergenceAnalyzer().analyze(contract)
print('Binomial order  ~', round(analysis['binomial_order'], 2))
print('Trinomial order ~', round(analysis['trinomial_order'], 2))
analysis['data'].round(5)

## 4. Volatility smile (SABR-generated, then re-fitted)

In [ ]:
F = 100 * np.exp(0.05)
sabr = SABRModel()
params = SABRParams(alpha=0.2, beta=1.0, rho=-0.3, nu=0.5)
strikes = np.linspace(70, 130, 13)
bsm = BlackScholesMerton()
quotes = []
for k in strikes:
    iv = sabr.implied_vol(F, k, 1.0, params.alpha, params.beta, params.rho, params.nu)
    quotes.append((float(k), float(bsm.price(100, k, 1.0, 0.05, iv, 'call')), 'call'))

smile = VolatilitySmile(quotes, S=100, T=1.0, r=0.05)
smile.extract_ivols().round(4)

## 5. Put-call parity & scenario analysis

In [ ]:
call = bsm.price(100, 100, 1.0, 0.05, 0.2, 'call')
put = bsm.price(100, 100, 1.0, 0.05, 0.2, 'put')
print('Parity:', PutCallParityCheck().check(call, put, 100, 100, 1.0, 0.05))

print('\nStress scenarios:')
display(StressTester().scenario_pnl(contract).round(4))

print('\nGreek P&L attribution (+5% spot, +2 vol pts, 1 day):')
GreekPnLAttribution().waterfall_data(contract, dS=5.0, dsigma=0.02, dT=1/365, dr=0.0).round(4)